# CIFAR-10 posterior-concentration diagnostics for closed-form flow

matching

This notebook accompanies the revised blog post on closed-form flow
matching and softmax collapse.

The goal is to analyze the closed-form weights

$$\lambda_i(x_t,t)
=
\frac{\exp\left(-\frac{\|x_t-tx^{(i)}\|^2}{2(1-t)^2}\right)}
{\sum_{j=1}^n \exp\left(-\frac{\|x_t-tx^{(j)}\|^2}{2(1-t)^2}\right)}.$$

The main change relative to the previous notebook is that the theory is
formulated as a posterior-decoding problem. With

$$r=\frac{t}{1-t},
\qquad
 y=\frac{x_t}{1-t},$$

the weights are

$$\lambda_i(y,r)
=
\frac{\exp\left(-\frac12\|y-rx^{(i)}\|^2\right)}
{\sum_{j=1}^n \exp\left(-\frac12\|y-rx^{(j)}\|^2\right)}.$$

Along a planted trajectory,

$$y=z+r x^{(i_\star)},
\qquad z\sim\mathcal N(0,I_d),$$

so the softmax is the posterior over which training sample generated the
observation.

The notebook computes:

- the isotropic posterior-decoding threshold;
- the covariance-aware log-det threshold when covariance eigenvalues are
  available;
- empirical diagnostics: planted mass, maximum mass, normalized entropy,
  and cosine alignment;
- a small Gaussian sanity check that runs without CIFAR-10;
- optional CIFAR-10 diagnostics, gated by a flag.

In [ ]:
# =============================================================================
# Imports and configuration
# =============================================================================
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.special import expit, logsumexp

# Make output directories relative to the notebook location.
FIGURES_DIR = Path('figures')
OUTPUT_DIR = Path('output')
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

np.set_printoptions(precision=6, suppress=True)


## Global parameters

The CIFAR-10 proxy used in the post is

$$d = 3\times 32\times 32 = 3072,
\qquad
n = 50,000,
\qquad
\sigma^2 \approx 0.24.$$

The value of (^2) is the per-coordinate variance after the standard
transformation from pixels in (\[0,1\]) to pixels in (\[-1,1\]).

In [ ]:
CIFAR10_D = 3 * 32 * 32
CIFAR10_N_FULL = 50_000
CIFAR10_SIGMA2 = 0.24
CIFAR10_ALPHA_FULL = np.log(CIFAR10_N_FULL) / CIFAR10_D

print(f'd = {CIFAR10_D}')
print(f'n = {CIFAR10_N_FULL}')
print(f'sigma^2 = {CIFAR10_SIGMA2:.4f}')
print(f'alpha = log(n)/d = {CIFAR10_ALPHA_FULL:.6f}')


## Posterior-decoding theory

The useful reparametrization is

$$r(t)=\frac{t}{1-t},
\qquad
 t(r)=\frac{r}{1+r}.$$

For the isotropic Gaussian proxy (x^{(i)}N(0,^2 I_d)), the information
rate is

$$\mathcal I_{\rm iso}(t)
=
\frac12\log\left(1+\sigma^2\left(\frac{t}{1-t}\right)^2\right).$$

The asymptotic threshold is

$$\mathcal I_{\rm iso}(t_C)=\frac{\log n}{d}.$$

Equivalently,

$$t_C
=
\frac{
\sqrt{(e^{2\alpha}-1)/\sigma^2}
}{
1+\sqrt{(e^{2\alpha}-1)/\sigma^2}
},
\qquad
\alpha=\frac{\log n}{d}.$$

In [ ]:
# =============================================================================
# Posterior-decoding theory functions
# =============================================================================

def alpha_from_n_d(n: int, d: int) -> float:
    """Return alpha = log(n) / d in nats per dimension."""
    if n <= 0 or d <= 0:
        raise ValueError('n and d must be positive')
    return float(np.log(n) / d)


def r_from_t(t):
    """Map t in [0, 1) to r = t / (1 - t). Supports scalars and arrays."""
    t_arr = np.asarray(t, dtype=float)
    if np.any(t_arr < 0) or np.any(t_arr >= 1):
        raise ValueError('t must belong to [0, 1)')
    out = t_arr / (1.0 - t_arr)
    return float(out) if out.ndim == 0 else out


def t_from_r(r):
    """Map r >= 0 to t = r / (1 + r). Supports scalars and arrays."""
    r_arr = np.asarray(r, dtype=float)
    if np.any(r_arr < 0):
        raise ValueError('r must be nonnegative')
    out = r_arr / (1.0 + r_arr)
    return float(out) if out.ndim == 0 else out


def info_rate_isotropic(t, sigma2: float):
    """
    Isotropic Gaussian information rate:
        I_iso(t) = 0.5 * log(1 + sigma2 * r(t)^2).
    """
    if sigma2 <= 0:
        raise ValueError('sigma2 must be positive')
    r = r_from_t(t)
    return 0.5 * np.log1p(sigma2 * np.asarray(r) ** 2)


def collapse_time_isotropic(n: int, d: int, sigma2: float) -> float:
    """Closed-form isotropic posterior-decoding threshold."""
    alpha = alpha_from_n_d(n, d)
    r_c = np.sqrt(np.expm1(2.0 * alpha) / sigma2)
    return t_from_r(r_c)


def planted_mass_approx_from_info(info_rate, n: int, d: int):
    """
    Finite-d logistic approximation to the planted posterior mass.

    lambda_star(t) approx 1 / (1 + exp(d * (alpha - I(t))))
    """
    alpha = alpha_from_n_d(n, d)
    exponent = d * (alpha - np.asarray(info_rate, dtype=float))
    return expit(-exponent)


def planted_mass_approx_isotropic(t, n: int, d: int, sigma2: float):
    """Finite-d planted-mass approximation under the isotropic Gaussian proxy."""
    return planted_mass_approx_from_info(info_rate_isotropic(t, sigma2), n, d)


def time_for_planted_mass_isotropic(
    target_mass: float,
    n: int,
    d: int,
    sigma2: float,
) -> float:
    """Solve the isotropic logistic approximation for lambda_star approx target_mass."""
    if not 0 < target_mass < 1:
        raise ValueError('target_mass must lie in (0, 1)')
    alpha = alpha_from_n_d(n, d)
    required_info = alpha + np.log(target_mass / (1.0 - target_mass)) / d
    if required_info <= 0:
        return 0.0
    r = np.sqrt(np.expm1(2.0 * required_info) / sigma2)
    return t_from_r(r)


In [ ]:
# CIFAR-10 isotropic proxy numbers

t_c_iso_full = collapse_time_isotropic(
    n=CIFAR10_N_FULL,
    d=CIFAR10_D,
    sigma2=CIFAR10_SIGMA2,
)

print(f'Isotropic posterior-decoding t_C: {t_c_iso_full:.6f}')
for mass in [0.5, 0.9, 0.99, 0.999]:
    t_mass = time_for_planted_mass_isotropic(
        target_mass=mass,
        n=CIFAR10_N_FULL,
        d=CIFAR10_D,
        sigma2=CIFAR10_SIGMA2,
    )
    print(f'Approximate t for lambda_star ~= {mass:>5}: {t_mass:.6f}')


## Covariance-aware theory

If (x^{(i)}N(0,C)) and (C) has eigenvalues (\_1,,\_d), the information
rate is

$$\mathcal I_C(t)
=
\frac{1}{2d}\log\det\left(I+r(t)^2 C\right)
=
\frac{1}{2d}\sum_{k=1}^d \log\left(1+r(t)^2\mu_k\right).$$

The covariance-aware threshold is

$$\mathcal I_C(t_C)=\frac{\log n}{d}.$$

In [ ]:
# =============================================================================
# Covariance-aware posterior-decoding functions
# =============================================================================

def info_rate_covariance(t, eigenvalues: np.ndarray):
    """
    Covariance-aware information rate:
        I_C(t) = (1/(2d)) sum_k log(1 + r(t)^2 * mu_k).

    Supports scalar or array t. Eigenvalues should be nonnegative.
    """
    eigs = np.asarray(eigenvalues, dtype=float)
    if eigs.ndim != 1 or eigs.size == 0:
        raise ValueError('eigenvalues must be a nonempty one-dimensional array')
    if np.any(eigs < -1e-12):
        raise ValueError('eigenvalues must be nonnegative')
    eigs = np.maximum(eigs, 0.0)

    t_arr = np.asarray(t, dtype=float)
    scalar_input = t_arr.ndim == 0
    t_flat = np.atleast_1d(t_arr)
    r = r_from_t(t_flat)
    values = 0.5 * np.mean(np.log1p((r[:, None] ** 2) * eigs[None, :]), axis=1)
    return float(values[0]) if scalar_input else values.reshape(t_arr.shape)


def collapse_time_covariance(n: int, eigenvalues: np.ndarray) -> float:
    """Solve I_C(t) = log(n)/d using covariance eigenvalues."""
    eigs = np.asarray(eigenvalues, dtype=float)
    d = eigs.size
    alpha = alpha_from_n_d(n, d)

    if np.all(eigs <= 0):
        return np.nan

    def equation(t: float) -> float:
        return info_rate_covariance(t, eigs) - alpha

    # At t = 0, I_C = 0. As t -> 1, I_C -> infinity if at least one eigenvalue is positive.
    return float(brentq(equation, 1e-12, 1.0 - 1e-10, maxiter=200))


def planted_mass_approx_covariance(t, n: int, eigenvalues: np.ndarray):
    """Finite-d planted-mass approximation using the covariance-aware information rate."""
    d = len(eigenvalues)
    return planted_mass_approx_from_info(info_rate_covariance(t, eigenvalues), n, d)


## Theory plots

The next plot shows the isotropic information rate and the
finite-dimensional planted-mass approximation for the CIFAR-10 proxy.

In [ ]:
def plot_isotropic_theory(
    n: int = CIFAR10_N_FULL,
    d: int = CIFAR10_D,
    sigma2: float = CIFAR10_SIGMA2,
    t_max: float = 0.5,
    path: Path | None = FIGURES_DIR / 'posterior_isotropic_theory.png',
):
    """Plot the isotropic information rate and planted-mass approximation."""
    alpha = alpha_from_n_d(n, d)
    t_grid = np.linspace(0.0, t_max, 500)
    info = info_rate_isotropic(t_grid, sigma2)
    mass = planted_mass_approx_from_info(info, n, d)
    t_c = collapse_time_isotropic(n, d, sigma2)

    plt.figure(figsize=(8, 5))
    plt.plot(t_grid, info, label='I_iso(t)')
    plt.axhline(alpha, linestyle='--', label='log(n)/d')
    plt.axvline(t_c, linestyle=':', label=f't_C={t_c:.3f}')
    plt.xlabel('time t')
    plt.ylabel('nats per dimension')
    plt.title('Isotropic posterior-decoding threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, dpi=160)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(t_grid, mass, label='logistic approximation to lambda_star(t)')
    plt.axvline(t_c, linestyle=':', label=f't_C={t_c:.3f}')
    plt.xlabel('time t')
    plt.ylabel('approximate planted posterior mass')
    plt.title('Finite-d planted-mass approximation')
    plt.ylim(-0.02, 1.02)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if path is not None:
        mass_path = path.with_name(path.stem + '_planted_mass.png')
        plt.savefig(mass_path, dpi=160)
    plt.show()


plot_isotropic_theory()


## Softmax and velocity diagnostics

For a planted trajectory, choose an index (i\_), draw (zN(0,I_d)), and
define

$$x_t=(1-t)z+t x^{(i_\star)}.$$

We compute four quantities:

$$\lambda_\star(t)=\lambda_{i_\star}(x_t,t),
\qquad
\lambda_{\max}(t)=\max_i\lambda_i(x_t,t),$$

$$\frac{H(\lambda(t))}{\log n},
\qquad
\cos_t
=
\cos\left(u_t(x_t), x^{(i_\star)}-z\right).$$

Here

$$u_t(x_t)=\sum_i\lambda_i(x_t,t)\frac{x^{(i)}-x_t}{1-t}
=\frac{\sum_i\lambda_i(x_t,t)x^{(i)}-x_t}{1-t}.$$

In [ ]:
# =============================================================================
# Exact empirical softmax diagnostics
# =============================================================================

def rowwise_cosine(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Rowwise cosine similarity between two arrays of shape (batch, d)."""
    numerator = np.sum(a * b, axis=1)
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return numerator / np.maximum(denom, eps)


def softmax_weights_for_batch(
    x_t: np.ndarray,
    data: np.ndarray,
    t: float,
    data_norms: np.ndarray | None = None,
) -> np.ndarray:
    """
    Compute exact softmax weights for a batch of query points.

    Uses the posterior form with y = x_t / (1 - t) and r = t / (1 - t):
        logits_i = -0.5 * ||y - r x_i||^2.
    """
    if not 0 <= t < 1:
        raise ValueError('t must belong to [0, 1)')

    data64 = np.asarray(data, dtype=np.float64)
    x_t64 = np.atleast_2d(np.asarray(x_t, dtype=np.float64))
    n, d = data64.shape
    if x_t64.shape[1] != d:
        raise ValueError('x_t and data have incompatible dimensions')

    if data_norms is None:
        data_norms = np.sum(data64 * data64, axis=1)
    else:
        data_norms = np.asarray(data_norms, dtype=np.float64)

    r = r_from_t(t)
    y = x_t64 / (1.0 - t)
    y_norms = np.sum(y * y, axis=1)

    # ||y - r x_i||^2 = ||y||^2 + r^2 ||x_i||^2 - 2 r <y, x_i>
    dist2 = y_norms[:, None] + (r * r) * data_norms[None, :] - 2.0 * r * (y @ data64.T)
    logits = -0.5 * dist2
    log_probs = logits - logsumexp(logits, axis=1, keepdims=True)
    weights = np.exp(log_probs)
    return weights


def compute_diagnostics_at_t(
    data: np.ndarray,
    t: float,
    n_queries: int = 64,
    rng: np.random.Generator | None = None,
    data_norms: np.ndarray | None = None,
) -> dict[str, float]:
    """Compute empirical posterior and velocity diagnostics at a single time."""
    if rng is None:
        rng = np.random.default_rng(0)

    data = np.asarray(data, dtype=np.float64)
    n, d = data.shape
    if n_queries <= 0:
        raise ValueError('n_queries must be positive')

    planted_indices = rng.integers(0, n, size=n_queries)
    x_star = data[planted_indices]
    z = rng.standard_normal(size=(n_queries, d))
    x_t = (1.0 - t) * z + t * x_star

    weights = softmax_weights_for_batch(x_t, data, t, data_norms=data_norms)
    posterior_mean = weights @ data
    marginal_velocity = (posterior_mean - x_t) / (1.0 - t)
    conditional_velocity = x_star - z

    row = np.arange(n_queries)
    lambda_star = weights[row, planted_indices]
    lambda_max = np.max(weights, axis=1)
    entropy = -np.sum(weights * np.log(np.maximum(weights, 1e-300)), axis=1)
    normalized_entropy = entropy / np.log(n)
    cosine = rowwise_cosine(marginal_velocity, conditional_velocity)
    argmax_is_star = np.argmax(weights, axis=1) == planted_indices

    return {
        't': float(t),
        'lambda_star_mean': float(np.mean(lambda_star)),
        'lambda_star_std': float(np.std(lambda_star)),
        'lambda_max_mean': float(np.mean(lambda_max)),
        'lambda_max_std': float(np.std(lambda_max)),
        'entropy_norm_mean': float(np.mean(normalized_entropy)),
        'entropy_norm_std': float(np.std(normalized_entropy)),
        'cosine_mean': float(np.mean(cosine)),
        'cosine_std': float(np.std(cosine)),
        'argmax_is_star_mean': float(np.mean(argmax_is_star)),
    }


def compute_diagnostics_vs_time(
    data: np.ndarray,
    t_values: np.ndarray,
    n_queries: int = 64,
    seed: int = 0,
) -> pd.DataFrame:
    """Compute diagnostics over a grid of time values."""
    data = np.asarray(data, dtype=np.float64)
    data_norms = np.sum(data * data, axis=1)
    rng = np.random.default_rng(seed)
    records = []
    for t in np.asarray(t_values, dtype=float):
        records.append(
            compute_diagnostics_at_t(
                data=data,
                t=float(t),
                n_queries=n_queries,
                rng=rng,
                data_norms=data_norms,
            )
        )
    return pd.DataFrame.from_records(records)


def plot_diagnostic_curves(
    results: pd.DataFrame,
    n: int,
    d: int,
    sigma2: float | None = None,
    eigenvalues: np.ndarray | None = None,
    title: str = 'Posterior diagnostics',
    path: Path | None = None,
):
    """Plot empirical diagnostics and optional theory thresholds."""
    plt.figure(figsize=(8, 5))
    plt.plot(results['t'], results['lambda_star_mean'], label='E[lambda_star]')
    plt.plot(results['t'], results['lambda_max_mean'], label='E[lambda_max]')
    plt.plot(results['t'], results['entropy_norm_mean'], label='E[H(lambda)/log n]')
    plt.plot(results['t'], results['cosine_mean'], label='E[cos_t]')

    if sigma2 is not None:
        t_c = collapse_time_isotropic(n=n, d=d, sigma2=sigma2)
        plt.axvline(t_c, linestyle=':', label=f'isotropic t_C={t_c:.3f}')

    if eigenvalues is not None:
        t_c_cov = collapse_time_covariance(n=n, eigenvalues=eigenvalues)
        plt.axvline(t_c_cov, linestyle='--', label=f'covariance t_C={t_c_cov:.3f}')

    plt.xlabel('time t')
    plt.ylabel('diagnostic value')
    plt.title(title)
    plt.ylim(-0.05, 1.05)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, dpi=160)
    plt.show()


## Small Gaussian sanity check

This section runs without CIFAR-10. It verifies the mechanics of the
posterior diagnostics on a smaller Gaussian codebook. The dimension and
sample size are intentionally small so that the notebook executes
quickly.

This plot is a computational sanity check, not a CIFAR-10 prediction.

In [ ]:
# =============================================================================
# Small synthetic sanity check
# =============================================================================

RUN_GAUSSIAN_SANITY_CHECK = True

if RUN_GAUSSIAN_SANITY_CHECK:
    rng = np.random.default_rng(123)
    d_demo = 128
    n_demo = 800
    sigma2_demo = 0.24
    data_demo = rng.normal(loc=0.0, scale=np.sqrt(sigma2_demo), size=(n_demo, d_demo))

    t_c_demo = collapse_time_isotropic(n=n_demo, d=d_demo, sigma2=sigma2_demo)
    print(f'Synthetic demo: n={n_demo}, d={d_demo}, sigma^2={sigma2_demo}')
    print(f'Synthetic isotropic threshold t_C={t_c_demo:.4f}')

    t_values_demo = np.linspace(0.0, 0.75, 31)
    demo_results = compute_diagnostics_vs_time(
        data=data_demo,
        t_values=t_values_demo,
        n_queries=32,
        seed=2026,
    )
    display(demo_results.head())

    plot_diagnostic_curves(
        results=demo_results,
        n=n_demo,
        d=d_demo,
        sigma2=sigma2_demo,
        title='Synthetic Gaussian codebook diagnostics',
        path=FIGURES_DIR / 'synthetic_posterior_diagnostics.png',
    )


## CIFAR-10 loading utilities

The following cells are optional. They are gated by
`RUN_CIFAR10_DIAGNOSTICS` because CIFAR-10 loading and full softmax
computations may be expensive.

Two consistency rules are important:

1.  If diagnostics are computed against only (n\_{}) images, the theory
    overlay should use ((n\_{})/d), not ((50,000)/d).
2.  If the plot is meant to describe the full training set, the softmax
    should be evaluated against all 50,000 images, possibly with fewer
    query trajectories and/or additional batching.

The loader uses Torchvision if available. If your environment already
contains CIFAR-10, set `download=False`; otherwise set `download=True`.

In [ ]:
# =============================================================================
# Optional CIFAR-10 loader and covariance utilities
# =============================================================================

def load_cifar10_data_numpy(
    n_samples: int | None = 1000,
    root: str | Path = './data',
    download: bool = True,
    random_subset: bool = False,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load CIFAR-10 training data with the standard normalization used in the post.

    Returns
    -------
    data : ndarray, shape (n_samples, 3072)
        Flattened images after ToTensor() and Normalize((0.5,...),(0.5,...)).
    labels : ndarray, shape (n_samples,)
        CIFAR-10 labels.
    """
    try:
        import torch
        from torchvision import datasets, transforms
    except Exception as exc:
        raise RuntimeError(
            'Could not import torch/torchvision. Install compatible versions, '
            'or run this notebook in an environment where CIFAR-10 can be loaded.'
        ) from exc

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    dataset = datasets.CIFAR10(
        root=str(root),
        train=True,
        download=download,
        transform=transform,
    )

    n_total = len(dataset)
    if n_samples is None:
        indices = np.arange(n_total)
    else:
        n_samples = min(int(n_samples), n_total)
        if random_subset:
            rng = np.random.default_rng(seed)
            indices = rng.choice(n_total, size=n_samples, replace=False)
        else:
            indices = np.arange(n_samples)

    data = np.empty((len(indices), 3 * 32 * 32), dtype=np.float32)
    labels = np.empty(len(indices), dtype=np.int64)

    for k, idx in enumerate(indices):
        image, label = dataset[int(idx)]
        data[k] = image.reshape(-1).detach().cpu().numpy()
        labels[k] = int(label)

    return data, labels


def compute_covariance_eigenvalues(data: np.ndarray, center: bool = True) -> np.ndarray:
    """Compute eigenvalues of the empirical covariance matrix of flattened data."""
    X = np.asarray(data, dtype=np.float64)
    if center:
        X = X - X.mean(axis=0, keepdims=True)
    n = X.shape[0]
    covariance = (X.T @ X) / n
    eigs = np.linalg.eigvalsh(covariance)
    eigs = np.maximum(eigs, 0.0)
    return np.sort(eigs)[::-1]


def load_or_compute_covariance_eigenvalues(
    data: np.ndarray,
    path: str | Path = OUTPUT_DIR / 'cifar10_cov_eigs.npy',
    force: bool = False,
) -> np.ndarray:
    """Load covariance eigenvalues from disk when available; otherwise compute and save."""
    path = Path(path)
    if path.exists() and not force:
        if path.suffix == '.npy':
            return np.load(path)
        return np.loadtxt(path)

    eigs = compute_covariance_eigenvalues(data)
    path.parent.mkdir(exist_ok=True, parents=True)
    if path.suffix == '.npy':
        np.save(path, eigs)
    else:
        np.savetxt(path, eigs)
    return eigs


def plot_covariance_spectrum(
    eigenvalues: np.ndarray,
    path: Path | None = FIGURES_DIR / 'cifar10_covariance_eigenvalues.png',
):
    """Plot the empirical covariance eigenvalue spectrum."""
    eigs = np.asarray(eigenvalues, dtype=float)

    plt.figure(figsize=(8, 5))
    plt.plot(eigs)
    plt.yscale('log')
    plt.xlabel('eigenvalue index')
    plt.ylabel('eigenvalue, log scale')
    plt.title('CIFAR-10 covariance eigenvalue spectrum')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, dpi=160)
    plt.show()


def plot_information_rate_with_covariance(
    n: int,
    sigma2: float,
    eigenvalues: np.ndarray,
    t_max: float = 0.8,
    path: Path | None = FIGURES_DIR / 'posterior_information_rate_covariance.png',
):
    """Plot isotropic and covariance-aware information rates against log(n)/d."""
    eigs = np.asarray(eigenvalues, dtype=float)
    d = len(eigs)
    alpha = alpha_from_n_d(n, d)
    t_grid = np.linspace(0.0, t_max, 600)
    info_iso = info_rate_isotropic(t_grid, sigma2=sigma2)
    info_cov = info_rate_covariance(t_grid, eigs)
    t_c_iso = collapse_time_isotropic(n=n, d=d, sigma2=sigma2)
    t_c_cov = collapse_time_covariance(n=n, eigenvalues=eigs)

    plt.figure(figsize=(8, 5))
    plt.plot(t_grid, info_iso, label='isotropic I_iso(t)')
    plt.plot(t_grid, info_cov, label='covariance I_C(t)')
    plt.axhline(alpha, linestyle='--', label='log(n)/d')
    plt.axvline(t_c_iso, linestyle=':', label=f'isotropic t_C={t_c_iso:.3f}')
    plt.axvline(t_c_cov, linestyle='-.', label=f'covariance t_C={t_c_cov:.3f}')
    plt.xlabel('time t')
    plt.ylabel('nats per dimension')
    plt.title('Posterior-decoding threshold with empirical covariance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, dpi=160)
    plt.show()


## Optional CIFAR-10 diagnostics

Set `RUN_CIFAR10_DIAGNOSTICS = True` to compute exact empirical softmax
diagnostics on CIFAR-10.

For quick local tests, start with `CIFAR10_N_FOR_SOFTMAX = 1000`. For
the figure used in the blog post, increase this toward 50,000 and reduce
`N_QUERY_TRAJECTORIES` if needed.

In [ ]:
# =============================================================================
# Optional CIFAR-10 empirical experiment
# =============================================================================

RUN_CIFAR10_DIAGNOSTICS = False

# For fast tests, use 1000. For the full training set, use None or 50_000.
CIFAR10_N_FOR_SOFTMAX = 1000
N_QUERY_TRAJECTORIES = 64
CIFAR10_T_VALUES = np.linspace(0.0, 0.5, 41)

if RUN_CIFAR10_DIAGNOSTICS:
    data_cifar, labels_cifar = load_cifar10_data_numpy(
        n_samples=CIFAR10_N_FOR_SOFTMAX,
        root='./data',
        download=True,
        random_subset=False,
        seed=0,
    )
    n_softmax, d_softmax = data_cifar.shape
    sigma2_empirical = float(np.mean(np.var(data_cifar, axis=0)))

    print(f'Loaded CIFAR-10 subset: n={n_softmax}, d={d_softmax}')
    print(f'Empirical average per-coordinate variance: {sigma2_empirical:.6f}')
    print(f'Using alpha=log({n_softmax})/{d_softmax}={alpha_from_n_d(n_softmax, d_softmax):.6f}')
    print(
        'Isotropic threshold for this softmax codebook: '
        f'{collapse_time_isotropic(n_softmax, d_softmax, sigma2_empirical):.6f}'
    )

    cifar_results = compute_diagnostics_vs_time(
        data=data_cifar,
        t_values=CIFAR10_T_VALUES,
        n_queries=N_QUERY_TRAJECTORIES,
        seed=2026,
    )
    display(cifar_results.head())

    cifar_results.to_csv(OUTPUT_DIR / 'cifar10_posterior_diagnostics.csv', index=False)

    plot_diagnostic_curves(
        results=cifar_results,
        n=n_softmax,
        d=d_softmax,
        sigma2=sigma2_empirical,
        title=f'CIFAR-10 posterior diagnostics, n={n_softmax}',
        path=FIGURES_DIR / 'cifar10_posterior_diagnostics.png',
    )
else:
    print('CIFAR-10 empirical diagnostics skipped. Set RUN_CIFAR10_DIAGNOSTICS = True to run them.')


## Optional CIFAR-10 covariance-aware threshold

Set `RUN_CIFAR10_COVARIANCE = True` to compute empirical covariance
eigenvalues and the log-det threshold.

For the covariance spectrum, it is preferable to use all 50,000 training
images. This computation is heavier than the subset softmax demo but
still feasible on a standard workstation.

In [ ]:
# =============================================================================
# Optional covariance-aware CIFAR-10 threshold
# =============================================================================

RUN_CIFAR10_COVARIANCE = False
CIFAR10_N_FOR_COVARIANCE = None  # None means all training images.

if RUN_CIFAR10_COVARIANCE:
    data_cov, _ = load_cifar10_data_numpy(
        n_samples=CIFAR10_N_FOR_COVARIANCE,
        root='./data',
        download=True,
        random_subset=False,
        seed=0,
    )
    eigs_cifar = load_or_compute_covariance_eigenvalues(
        data_cov,
        path=OUTPUT_DIR / 'cifar10_cov_eigs.npy',
        force=False,
    )

    print(f'Computed/loaded {len(eigs_cifar)} covariance eigenvalues')
    print(f'Mean eigenvalue: {np.mean(eigs_cifar):.6f}')
    print(f'Max eigenvalue: {np.max(eigs_cifar):.6f}')
    print(f'Min eigenvalue: {np.min(eigs_cifar):.6e}')

    plot_covariance_spectrum(eigs_cifar)

    t_c_cov_full = collapse_time_covariance(CIFAR10_N_FULL, eigs_cifar)
    print(f'Covariance-aware threshold for full CIFAR-10 n=50,000: {t_c_cov_full:.6f}')

    plot_information_rate_with_covariance(
        n=CIFAR10_N_FULL,
        sigma2=CIFAR10_SIGMA2,
        eigenvalues=eigs_cifar,
        t_max=0.8,
        path=FIGURES_DIR / 'posterior_information_rate_covariance.png',
    )
else:
    print('CIFAR-10 covariance computation skipped. Set RUN_CIFAR10_COVARIANCE = True to run it.')


## Summary table

The following cell summarizes the theory values that do not require
loading CIFAR-10.

In [ ]:
summary = pd.DataFrame([
    {
        'quantity': 'alpha = log(50000)/3072',
        'value': CIFAR10_ALPHA_FULL,
    },
    {
        'quantity': 'isotropic t_C, sigma^2=0.24',
        'value': t_c_iso_full,
    },
    {
        'quantity': 'isotropic t for lambda_star approx 0.9',
        'value': time_for_planted_mass_isotropic(0.9, CIFAR10_N_FULL, CIFAR10_D, CIFAR10_SIGMA2),
    },
    {
        'quantity': 'isotropic t for lambda_star approx 0.99',
        'value': time_for_planted_mass_isotropic(0.99, CIFAR10_N_FULL, CIFAR10_D, CIFAR10_SIGMA2),
    },
])
summary


## Notes for blog-post integration

Recommended figures to export from this notebook:

1.  `figures/posterior_isotropic_theory.png`: isotropic information-rate
    threshold.
2.  `figures/posterior_isotropic_theory_planted_mass.png`:
    finite-dimensional planted-mass approximation.
3.  `figures/cifar10_posterior_diagnostics.png`: empirical posterior
    diagnostics, if CIFAR-10 is run.
4.  `figures/posterior_information_rate_covariance.png`:
    covariance-aware threshold, if covariance eigenvalues are computed.
5.  `figures/cifar10_covariance_eigenvalues.png`: empirical covariance
    spectrum.

The key consistency check is that the value of (n) in (n/d) should match
the number of training samples used in the softmax denominator.